# KPI Analysis

## Purpose
This notebook runs the 12 KPI queries written for this project, checks that their results agree with each other, charts them, and records which differences between groups are large enough to take seriously. The goal is to answer the problem statement about when and why calls go unanswered. The analysis is descriptive only, so it explains what happened in the data and does not predict anything.

## Scope
The notebook covers 12 KPIs, which come from 4 metrics measured across several groupings:

- **Unanswered call rate** by hour of day, month, agent and department
- **Average handling time (AHT)** by agent and department. In this dataset, AHT means average talk duration only, because there is no hold or after-call work data.
- **Average satisfaction rating** (scale of 1 to 5) by agent and department
- **Average speed of answer** by hour of day, month, agent and department

The hourly KPIs leave out the 18:00 hour, which contains only 14 calls. All other KPIs use every call in the dataset.

## Source
- **Database:** `data/database/call_center.db`
- **View queried by every KPI:** `vw_call_records_enriched`, created by `sql/00_create_view.sql`
- **KPI queries:** `sql/kpis/`

## How to run
1. Make sure the database exists and the view has been created by running `sql/00_create_view.sql` against it.
2. Choose Restart & Run All. The reconciliation checks near the top stop the notebook with an error if any query returns numbers that do not add up.

## Structure
1. Setup
2. Reconciliation checks
3. One section per metric, each with tables, charts and a short observation
4. Findings# KPI Analysis

## Purpose
This notebook runs the 12 KPI queries written for this project, checks that their results agree with each other, charts them, and records which differences between groups are large enough to take seriously. The goal is to answer the problem statement about when and why calls go unanswered. The analysis is descriptive only, so it explains what happened in the data and does not predict anything.

## Scope
The notebook covers 12 KPIs, which come from 4 metrics measured across several groupings:

- **Unanswered call rate** by hour of day, month, agent and department
- **Average handling time (AHT)** by agent and department. In this dataset, AHT means average talk duration only, because there is no hold or after-call work data.
- **Average satisfaction rating** (scale of 1 to 5) by agent and department
- **Average speed of answer** by hour of day, month, agent and department

The hourly KPIs leave out the 18:00 hour, which contains only 14 calls. All other KPIs use every call in the dataset.

## Source
- **Database:** `data/database/call_center.db`
- **View queried by every KPI:** `vw_call_records_enriched`, created by `sql/00_create_view.sql`
- **KPI queries:** `sql/kpis/`

## How to run
1. The database `data/database/call_center.db` must exist (it is produced by `01_cleaning.ipynb`). The setup cell recreates the view *vw_call_records_enriched* from `sql/00_create_view.sql` automatically.
2. Choose Restart & Run All. The reconciliation checks near the top stop the notebook with an error if any query returns numbers that do not add up.

## Structure
1. Setup
2. Reconciliation checks
3. One section per metric, each with tables, charts and a short observation
4. Findings

## Setup

In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd
import sqlparse

In [51]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [52]:
db_name = 'call_center.db'
src_path = Path.cwd().parent / "data/database" / db_name
assert src_path.exists(), "Database does not exist"
conn = sqlite3.connect(src_path)

In [53]:
kpi_mapping = {
    "unanswered_rate_month": "kpi_unanswered_pct_by_month.sql",
    "unanswered_rate_hour": "kpi_unanswered_pct_by_hour.sql",
    "unanswered_rate_agnt": "kpi_unanswered_pct_by_agnt.sql",
    "unanswered_rate_dept": "kpi_unanswered_pct_by_dept.sql",
    "aht_agnt": "kpi_aht_by_agnt.sql",
    "aht_dept": "kpi_aht_by_dept.sql",
    "avg_satisfaction_agnt": "kpi_avg_satisfaction_rating_by_agnt.sql",
    "avg_satisfaction_dept": "kpi_avg_satisfaction_rating_by_dept.sql",
    "avg_soa_month": "kpi_avg_soa_by_month.sql",
    "avg_soa_hour": "kpi_avg_soa_by_hour.sql",
    "avg_soa_agnt": "kpi_avg_soa_by_agnt.sql",
    "avg_soa_dept": "kpi_avg_soa_by_dept.sql",
}

In [54]:
kpi_src_dir = Path.cwd().parent / "sql/kpis"
assert set(kpi_mapping.values()) == {p.name for p in kpi_src_dir.glob('kpi_*.sql')}

In [55]:
create_view_script = Path.cwd().parent / "sql" / "00_create_view.sql"
cursor = conn.cursor()
with open(create_view_script, "r", encoding="utf-8") as file:
    sql_content = file.read()
    queries = sqlparse.split(sql_content)
    for query in queries:
        cursor.execute(query)

conn.commit()

In [56]:
def run_kpi(kpi_name, conn):

    filename = kpi_mapping[kpi_name]
    filepath = kpi_src_dir / filename

    with open(filepath, "r", encoding="utf-8") as file:
        sql_content = file.read()

    queries = sqlparse.split(sql_content)
    assert len(queries) == 1, filename

    return {
        "filename": filename,
        "sql_query": queries[0],
        "dataframe": pd.read_sql_query(queries[0], conn),
    }

In [57]:
sql_results = {name: run_kpi(name, conn) for name in kpi_mapping}

In [58]:
conn.close()

## Reconciliation

In [59]:
dataframes = {name: result['dataframe'] for name, result in sql_results.items()}

### Row counts

In [ ]:
df = dataframes["unanswered_rate_month"]
row_count = len(df)
assert row_count == 3, (
    f"Row counts for Unanswered rate by month KPI is {row_count}, expected 3"
)

df = dataframes["unanswered_rate_hour"]
row_count = len(df)
assert row_count == 9, (
    f"Row counts for Unanswered rate by hour KPI is {row_count}, expected 9"
)

df = dataframes["unanswered_rate_agnt"]
row_count = len(df)
assert row_count == 8, (
    f"Row counts for Unanswered rate by agent KPI is {row_count}, expected 8"
)

df = dataframes["unanswered_rate_dept"]
row_count = len(df)
assert row_count == 5, (
    f"Row counts for Unanswered rate by department KPI is {row_count}, expected 5"
)

df = dataframes["aht_agnt"]
row_count = len(df)
assert row_count == 8, f"Row counts for AHT by agent KPI is {row_count}, expected 8"

df = dataframes["aht_dept"]
row_count = len(df)
assert row_count == 5, (
    f"Row counts for AHT by department KPI is {row_count}, expected 5"
)

df = dataframes["avg_satisfaction_agnt"]
row_count = len(df)
assert row_count == 8, (
    f"Row counts for Average satisfaction rating by agent KPI is {row_count}, expected 8"
)

df = dataframes["avg_satisfaction_dept"]
row_count = len(df)
assert row_count == 5, (
    f"Row counts for Average satisfaction rating by department KPI is {row_count}, expected 5"
)

df = dataframes["avg_soa_month"]
row_count = len(df)
assert row_count == 3, (
    f"Row counts for Average speed of answer by month KPI is {row_count}, expected 3"
)

df = dataframes["avg_soa_hour"]
row_count = len(df)
assert row_count == 9, (
    f"Row counts for Average speed of answer by hour KPI is {row_count}, expected 9"
)

df = dataframes["avg_soa_agnt"]
row_count = len(df)
assert row_count == 8, (
    f"Row counts for Average speed of answer by agent KPI is {row_count}, expected 8"
)

df = dataframes["avg_soa_dept"]
row_count = len(df)
assert row_count == 5, (
    f"Row counts for Average speed of answer by department KPI is {row_count}, expected 5"
)


### Total calls

In [ ]:
df = dataframes["unanswered_rate_month"]
total_calls = df["total_call_cnt"].sum()
assert total_calls == 5000, (
    f"Total call count for Unanswered rate by month KPI is {total_calls}, expected 5000"
)

df = dataframes["unanswered_rate_hour"]
total_calls = df["total_call_cnt"].sum()
assert total_calls == 4986, (
    f"Total call count for Unanswered rate by month KPI is {total_calls}, expected 4986"
)

df = dataframes["unanswered_rate_agnt"]
total_calls = df["assigned_call_cnt"].sum()
assert total_calls == 5000, (
    f"Total call count for Unanswered rate by agnt KPI is {total_calls}, expected 5000"
)

df = dataframes["unanswered_rate_dept"]
total_calls = df["total_call_cnt"].sum()
assert total_calls == 5000, (
    f"Total call count for Unanswered rate by dept KPI is {total_calls}, expected 5000"
)


### Unanswered calls

In [ ]:
df = dataframes["unanswered_rate_month"]
call_count = df["unanswered_call_cnt"].sum()
assert call_count == 946, (
    f"Unanswered call counts for Unanswered rate by month KPI is {call_count}, expected 946"
)

df = dataframes["unanswered_rate_hour"]
call_count = df["unanswered_call_cnt"].sum()
assert 932 <= call_count <= 946, (
    f"Unanswered call counts for Unanswered rate by hour KPI is {call_count}, expected between 932 and 946"
)

df = dataframes["unanswered_rate_agnt"]
call_count = df["unanswered_call_cnt"].sum()
assert call_count == 946, (
    f"Unanswered call counts for Unanswered rate by agent KPI is {call_count}, expected 946"
)

df = dataframes["unanswered_rate_dept"]
call_count = df["unanswered_call_cnt"].sum()
assert call_count == 946, (
    f"Unanswered call counts for Unanswered rate by department KPI is {call_count}, expected 946"
)


### Answered calls

In [63]:
df = dataframes['aht_agnt']
call_count = df['answered_call_cnt'].sum()
assert call_count == 4054, f"Call counts for AHT by agent KPI is {call_count}, expected 4054"

df = dataframes['aht_dept']
call_count = df['answered_call_cnt'].sum()
assert call_count == 4054, f"Call counts for AHT by department KPI is {call_count}, expected 4054"

df = dataframes['avg_satisfaction_agnt']
call_count = df['answered_call_cnt'].sum()
assert call_count == 4054, f"Call counts for Average satisfaction by agent KPI is {call_count}, expected 4054"

df = dataframes['avg_satisfaction_dept']
call_count = df['answered_call_cnt'].sum()
assert call_count == 4054, f"Call counts for Average satisfaction by department KPI is {call_count}, expected 4054"

df = dataframes['avg_soa_month']
call_count = df['answered_call_cnt'].sum()
assert call_count == 4054, f"Call counts for Average speed of answer by month KPI is {call_count}, expected 4054"

df = dataframes['avg_soa_agnt']
call_count = df['answered_call_cnt'].sum()
assert call_count == 4054, f"Call counts for Average speed of answer by agent KPI is {call_count}, expected 4054"

df = dataframes['avg_soa_dept']
call_count = df['answered_call_cnt'].sum()
assert call_count == 4054, f"Call counts for Average speed of answer by department KPI is {call_count}, expected 4054"


### Speed of answer total

In [ ]:
df = dataframes['avg_soa_month']
soa_total = df['total_speed_of_answer_sec'].sum()
assert soa_total == 273729, f"Total speed of answer for Average speed of answer by month KPI is {soa_total}, expected 273,729"

df = dataframes['avg_soa_agnt']
soa_total = df['total_speed_of_answer_sec'].sum()
assert soa_total == 273729, f"Total speed of answer for Average speed of answer by agent KPI is {soa_total}, expected 273,729"

df = dataframes['avg_soa_dept']
soa_total = df['total_speed_of_answer_sec'].sum()
assert soa_total == 273729, f"Total speed of answer for Average speed of answer by department KPI is {soa_total}, expected 273,729"


### Weighted means

In [65]:
df = dataframes["aht_agnt"]
weighted_mean = (df["aht_sec"] * df["answered_call_cnt"]).sum() / df[
    "answered_call_cnt"
].sum()
assert abs(weighted_mean - 224.42) < 0.01, f"weighted mean was {weighted_mean}"

df = dataframes["aht_dept"]
weighted_mean = (df["aht_sec"] * df["answered_call_cnt"]).sum() / df[
    "answered_call_cnt"
].sum()
assert abs(weighted_mean - 224.42) < 0.01, f"weighted mean was {weighted_mean}"

df = dataframes["avg_satisfaction_agnt"]
weighted_mean = (df["avg_satisfaction_rating"] * df["answered_call_cnt"]).sum() / df[
    "answered_call_cnt"
].sum()
assert abs(weighted_mean - 3.40) < 0.01, f"weighted mean was {weighted_mean}"

df = dataframes["avg_satisfaction_dept"]
weighted_mean = (df["avg_satisfaction_rating"] * df["answered_call_cnt"]).sum() / df[
    "answered_call_cnt"
].sum()
assert abs(weighted_mean - 3.40) < 0.01, f"weighted mean was {weighted_mean}"

df = dataframes["avg_soa_month"]
weighted_mean = (df["avg_speed_of_answer_sec"] * df["answered_call_cnt"]).sum() / df[
    "answered_call_cnt"
].sum()
assert abs(weighted_mean - 67.52) < 0.01, f"weighted mean was {weighted_mean}"

df = dataframes["avg_soa_agnt"]
weighted_mean = (df["avg_speed_of_answer_sec"] * df["answered_call_cnt"]).sum() / df[
    "answered_call_cnt"
].sum()
assert abs(weighted_mean - 67.52) < 0.01, f"weighted mean was {weighted_mean}"

df = dataframes["avg_soa_dept"]
weighted_mean = (df["avg_speed_of_answer_sec"] * df["answered_call_cnt"]).sum() / df[
    "answered_call_cnt"
].sum()
assert abs(weighted_mean - 67.52) < 0.01, f"weighted mean was {weighted_mean}"